# CS570 Week 2 Lab: HDFS & Distributed Thinking

**Instructor:** Dr. Ragnar Lesch  
**Duration:** 60 minutes  
**Date:** Week 2, Spring 2026

---

## Instructions

This notebook contains **three parts**:

- **Part 1 (20 min):** HDFS CLI commands
- **Part 2 (40 min):** PySpark coding
- **Part 3:** Reflection questions

**How to work through this notebook:**
1. Run code cells in order using `Shift+Enter`
2. Answer questions in the markdown cells provided (double-click to edit)
3. Shell commands use `!` prefix — output is captured in the notebook

**Prerequisites:**
- HDFS is running (`jps` should show NameNode and DataNode)
- Python 3.11 and PySpark 4.1.x installed

**Submission:** Export this notebook as PDF when complete.

---

# Part 1: HDFS Command Line Interface

**Time:** 20 minutes

In this section, you'll use HDFS commands to understand distributed storage concepts.

## Exercise 1.1: Setup Verification (3 minutes)

First, verify your environment is working correctly.

In [1]:
# Check Hadoop installation
!hadoop version

Hadoop 3.4.2
Source code repository https://github.com/apache/hadoop.git -r 84e8b89ee2ebe6923691205b9e171badde7a495c
Compiled by ahmarsu on 2025-08-20T10:30Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum fa94c67d4b4be021b9e9515c9b0f7b6
This command was run using /opt/homebrew/Cellar/hadoop/3.4.2/libexec/share/hadoop/common/hadoop-common-3.4.2.jar


In [2]:
# Check HDFS is running
!hdfs dfs -ls /

'hdfs' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
# Check PySpark
import pyspark
print(f"PySpark version: {pyspark.__version__}")

PySpark version: 4.1.1


---

## Exercise 1.2: Create and Upload Data to HDFS (5 minutes)

### Step 1: Create your HDFS workspace

In [4]:
# Create a directory for this week's work
!hdfs dfs -mkdir -p /user/$(whoami)/week2

### Step 2: Create sample data locally

In [5]:
# Create a local text file with e-commerce transaction data
sample_data = """customer_001,laptop,1299.99,San Francisco,completed
customer_002,headphones,89.99,New York,completed
customer_001,mouse,24.99,San Francisco,completed
customer_003,keyboard,159.99,Los Angeles,pending
customer_002,monitor,449.99,New York,completed"""

with open("sample.txt", "w") as f:
    f.write(sample_data)

print("Created sample.txt")
print("\nContents:")
print(sample_data)

Created sample.txt

Contents:
customer_001,laptop,1299.99,San Francisco,completed
customer_002,headphones,89.99,New York,completed
customer_001,mouse,24.99,San Francisco,completed
customer_003,keyboard,159.99,Los Angeles,pending
customer_002,monitor,449.99,New York,completed


### Step 3: Upload to HDFS

In [6]:
# Upload file to HDFS
!hdfs dfs -put -f sample.txt /user/$(whoami)/week2/

# Verify upload
!hdfs dfs -ls /user/$(whoami)/week2/

Found 2 items
-rw-r--r--   1 leschrh69 supergroup        245 2026-01-19 23:47 /user/leschrh69/week2/sample.txt
-rw-r--r--   1 leschrh69 supergroup        307 2026-01-18 19:09 /user/leschrh69/week2/weather_data.txt


### Step 4: Check file metadata

In [7]:
# Check file status and replication
# Format: file_size block_size replication_factor filename
!hdfs dfs -stat "%b %o %r %n" /user/$(whoami)/week2/sample.txt

245 134217728 1 sample.txt


---

### Question 1

**What is the replication factor of your file? Why is it set to 1 instead of the default 3?**

*YOUR ANSWER:*



---

## Exercise 1.3: Understanding Block Storage (5 minutes)

### Step 1: Examine block details

In [8]:
# Check block locations and health
!hdfs fsck /user/$(whoami)/week2/sample.txt -files -blocks -locations

Connecting to namenode via http://localhost:9870/fsck?ugi=leschrh69&files=1&blocks=1&locations=1&path=%2Fuser%2Fleschrh69%2Fweek2%2Fsample.txt
FSCK started by leschrh69 (auth:SIMPLE) from /127.0.0.1 for path /user/leschrh69/week2/sample.txt at Mon Jan 19 23:47:30 PST 2026

/user/leschrh69/week2/sample.txt 245 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-2018195647-127.0.0.1-1768678368272:blk_1073741828_1004 len=245 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-69182739-8bb9-4453-a63d-7a9c9f56932d,DISK]]


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			0
 Total symlinks:		0

Replicated Blocks:
 Total size:	245 B
 Total files:	1
 Total blocks (validated):	1 (avg. block size 245 B)
 Minimally replicated blocks:	1 (100.0 %)
 Over-replicated blocks:	0 (0.0 %)
 Under-replicated blocks:	0 (0.0 %)
 Mis-replicated blocks:		0 (0.0 %)
 Default replication factor:	1
 Average block replication:	1.0
 Missing blocks:		0
 Corrupt blocks:		0
 Missing r

---

### Question 2

**Answer these based on your `fsck` output:**

a) How many blocks does your file have?

b) On which DataNode is the block stored? (IP address and port)

c) How many racks does your cluster have?

d) Why do you only see 1 DataNode instead of multiple DataNodes?

*YOUR ANSWERS:*

a) 

b) 

c) 

d) 


---

## Exercise 1.4: Replication in Action (4 minutes)

### Step 1: Increase replication factor

In [9]:
# Change replication from 1 to 2
!hdfs dfs -setrep 2 /user/$(whoami)/week2/sample.txt

Replication 2 set: /user/leschrh69/week2/sample.txt


### Step 2: Verify the change

In [10]:
# Check new replication factor
!hdfs dfs -stat "%r" /user/$(whoami)/week2/sample.txt

# See updated block locations
!hdfs fsck /user/$(whoami)/week2/sample.txt -files -blocks -locations

2
Connecting to namenode via http://localhost:9870/fsck?ugi=leschrh69&files=1&blocks=1&locations=1&path=%2Fuser%2Fleschrh69%2Fweek2%2Fsample.txt
FSCK started by leschrh69 (auth:SIMPLE) from /127.0.0.1 for path /user/leschrh69/week2/sample.txt at Mon Jan 19 23:47:38 PST 2026

/user/leschrh69/week2/sample.txt 245 bytes, replicated: replication=2, 1 block(s):  Under replicated BP-2018195647-127.0.0.1-1768678368272:blk_1073741828_1004. Target Replicas is 2 but found 1 live replica(s), 0 decommissioned replica(s), 0 decommissioning replica(s).
0. BP-2018195647-127.0.0.1-1768678368272:blk_1073741828_1004 len=245 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-69182739-8bb9-4453-a63d-7a9c9f56932d,DISK]]


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			0
 Total symlinks:		0

Replicated Blocks:
 Total size:	245 B
 Total files:	1
 Total blocks (validated):	1 (avg. block size 245 B)
 Minimally replicated blocks:	1 (100.0 %)
 Over-replicated blocks:	0 (0.0 %)


---

### Question 3

**What changed after setting replication to 2?**

a) What replication factor does `stat` show now?

b) On a single-machine setup, how many actual copies exist? Why?

c) What would happen on a 3-node cluster with replication set to 2?

*YOUR ANSWERS:*

a) 

b) 

c) 


---

## Exercise 1.5: Data Locality Verification (3 minutes)

### Step 1: Download from HDFS

In [11]:
# Download file back to local system
!hdfs dfs -get /user/$(whoami)/week2/sample.txt downloaded_sample.txt
print("Downloaded file from HDFS")

Downloaded file from HDFS


### Step 2: Compare files

In [12]:
# Compare original and downloaded (no output = identical)
!diff sample.txt downloaded_sample.txt && echo "Files are identical"

Files are identical


---

### Question 4

**Why does HDFS try to run Map tasks on the same machine that stores the data blocks?**

*YOUR ANSWER:*



---

## Exercise 1.6: Under the Hood (Optional - 5 minutes)

### Part A: See Where HDFS Actually Stores Your Data

HDFS stores blocks as regular files on your local disk. Let's find them:

In [13]:
# Get the configured data directory
!hdfs getconf -confKey dfs.datanode.data.dir

file:///tmp/hadoop-leschrh69/dfs/data


In [14]:
import subprocess

# Get the data directory path
result = subprocess.run(['hdfs', 'getconf', '-confKey', 'dfs.datanode.data.dir'], 
                       capture_output=True, text=True)
data_dir = result.stdout.strip().replace('file://', '')
print(f"Data directory: {data_dir}")

# Find block files
print("\nBlock files found:")
!find {data_dir} -name "blk_*" -type f 2>/dev/null | head -10

Data directory: /tmp/hadoop-leschrh69/dfs/data

Block files found:
/tmp/hadoop-leschrh69/dfs/data/current/BP-2018195647-127.0.0.1-1768678368272/current/finalized/subdir0/subdir0/blk_1073741828_1004.meta
/tmp/hadoop-leschrh69/dfs/data/current/BP-2018195647-127.0.0.1-1768678368272/current/finalized/subdir0/subdir0/blk_1073741827
/tmp/hadoop-leschrh69/dfs/data/current/BP-2018195647-127.0.0.1-1768678368272/current/finalized/subdir0/subdir0/blk_1073741828
/tmp/hadoop-leschrh69/dfs/data/current/BP-2018195647-127.0.0.1-1768678368272/current/finalized/subdir0/subdir0/blk_1073741827_1003.meta


In [15]:
# Read the actual block file - it contains your data!
import subprocess
result = subprocess.run(['hdfs', 'getconf', '-confKey', 'dfs.datanode.data.dir'], 
                       capture_output=True, text=True)
data_dir = result.stdout.strip().replace('file://', '')

print("Contents of a block file:")
!cat {data_dir}/current/BP-*/current/finalized/subdir0/subdir0/blk_1073741*[!.meta] 2>/dev/null | head

Contents of a block file:
San Francisco,2024-01-01,58
San Francisco,2024-01-02,62
San Francisco,2024-01-03,55
New York,2024-01-01,32
New York,2024-01-02,28
New York,2024-01-03,35
Los Angeles,2024-01-01,72
Los Angeles,2024-01-02,75
Los Angeles,2024-01-03,70
San Francisco,2024-01-04,60


---

### Question 5 (Optional)

**HDFS shows your file at `/user/$(whoami)/week2/sample.txt`, but physically it's stored as block files in `/tmp/hadoop-.../dfs/data/...`. Why does HDFS use this abstraction instead of just using regular file paths? What benefits does it provide?**

*YOUR ANSWER:*



### Part B: HDFS Data Persistence

**⚠️ Run these commands in a separate terminal window, not in the notebook.** Stopping HDFS while the notebook is running can cause connection issues.

```bash
# In a terminal window (not the notebook):

# Stop HDFS services
stop-dfs.sh

# HDFS commands won't work (no NameNode running)
hdfs dfs -ls /user/$(whoami)/week2/
# This will fail with "Connection refused"

# But the actual data files still exist on disk!
find /tmp/hadoop-$(whoami)/dfs/data -name "blk_*" 2>/dev/null | head -3

# Restart HDFS
start-dfs.sh
jps  # Verify NameNode and DataNode are running

# Now HDFS commands work again
hdfs dfs -ls /user/$(whoami)/week2/
```

---

### Question 6 (Optional)

**In a 3-node cluster with replication=3:**
- What happens if 1 DataNode stops? Can you still access your files?
- What happens if 2 DataNodes stop? Can you still access your files?
- What happens if the NameNode stops? What about the data?

*Hint: Think about the difference between metadata (file locations) and actual data (blocks).*

*YOUR ANSWER:*



---

# Part 2: Max Temperature Problem with PySpark

**Time:** 40 minutes

Now we'll use Spark to analyze data stored in HDFS.

## Exercise 2.1: Create Weather Dataset

First, we'll create sample weather data and upload it to HDFS.

In [16]:
# Create weather data file locally
weather_data = """San Francisco,2024-01-01,58
San Francisco,2024-01-02,62
San Francisco,2024-01-03,55
New York,2024-01-01,32
New York,2024-01-02,28
New York,2024-01-03,35
Los Angeles,2024-01-01,72
Los Angeles,2024-01-02,75
Los Angeles,2024-01-03,70
San Francisco,2024-01-04,60
New York,2024-01-04,30
Los Angeles,2024-01-04,73"""

# Write to local file
with open('weather_data.txt', 'w') as f:
    f.write(weather_data)
    
print("✓ Created weather_data.txt with 12 records")
print("\nSample data:")
print(weather_data[:150] + "...")

✓ Created weather_data.txt with 12 records

Sample data:
San Francisco,2024-01-01,58
San Francisco,2024-01-02,62
San Francisco,2024-01-03,55
New York,2024-01-01,32
New York,2024-01-02,28
New York,2024-01-03,...


In [17]:
# Upload to HDFS
!hdfs dfs -put -f weather_data.txt /user/$(whoami)/week2/

# Verify upload
print("HDFS directory contents:")
!hdfs dfs -ls /user/$(whoami)/week2/

HDFS directory contents:
Found 2 items
-rw-r--r--   2 leschrh69 supergroup        245 2026-01-19 23:47 /user/leschrh69/week2/sample.txt
-rw-r--r--   1 leschrh69 supergroup        307 2026-01-19 23:49 /user/leschrh69/week2/weather_data.txt


---

## Exercise 2.2: Initialize Spark and Load Data

**Key Concepts:**
- `SparkSession` - entry point to Spark functionality
- `.master("local[*]")` - run locally using all available cores
- Reading from HDFS requires the full `hdfs://` URL

In [18]:
from pyspark.sql import SparkSession
import os

# Initialize Spark
spark = SparkSession.builder \
    .appName("MaxTemperature") \
    .master("local[*]") \
    .getOrCreate()

print("✓ Spark initialized")
print(f"  Spark version: {spark.version}")
print(f"  App name: {spark.sparkContext.appName}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/19 23:49:30 WARN Utils: Your hostname, News-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.130 instead (on interface en0)
26/01/19 23:49:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/19 23:49:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✓ Spark initialized
  Spark version: 4.1.1
  App name: MaxTemperature


In [19]:
# Read CSV file from HDFS
username = os.environ.get('USER', 'your_username')
hdfs_file_path = f"hdfs://localhost:9000/user/{username}/week2/weather_data.txt"

df = spark.read.csv(
    hdfs_file_path,
    header=False,
    inferSchema=True
)

print("✓ Data loaded from HDFS")
print(f"  Number of rows: {df.count()}")
print(f"  Number of columns: {len(df.columns)}")

✓ Data loaded from HDFS
  Number of rows: 12
  Number of columns: 3


In [20]:
# Rename columns for clarity
df = df.toDF("city", "date", "temperature")

print("✓ Columns renamed")
print("\nDataFrame schema:")
df.printSchema()

print("\nSample data:")
df.show(5)

✓ Columns renamed

DataFrame schema:
root
 |-- city: string (nullable = true)
 |-- date: date (nullable = true)
 |-- temperature: integer (nullable = true)


Sample data:
+-------------+----------+-----------+
|         city|      date|temperature|
+-------------+----------+-----------+
|San Francisco|2024-01-01|         58|
|San Francisco|2024-01-02|         62|
|San Francisco|2024-01-03|         55|
|     New York|2024-01-01|         32|
|     New York|2024-01-02|         28|
+-------------+----------+-----------+
only showing top 5 rows


---

## Exercise 2.3: Find Maximum Temperature Per City

This is where MapReduce thinking comes in!

**Think about it:**
- **Map:** We already have (city, temperature) pairs in our DataFrame
- **Shuffle:** `groupBy("city")` groups all records by city
- **Reduce:** `max("temperature")` finds the maximum for each group

In [21]:
from pyspark.sql import functions as F

# Find maximum temperature per city
max_temps = df.groupBy("city").agg(
    F.max("temperature").alias("max_temperature")
)

print("✓ Calculated maximum temperatures")
print("\nResults:")
max_temps.show()

✓ Calculated maximum temperatures

Results:
+-------------+---------------+
|         city|max_temperature|
+-------------+---------------+
|  Los Angeles|             75|
|San Francisco|             62|
|     New York|             35|
+-------------+---------------+



---

### Question 7

**In the MapReduce model, what work happens during each phase for the Spark code above?**

a) **Map phase:** What transformation happens to the input data?

b) **Shuffle phase:** What data movement is required and why?

c) **Reduce phase:** What aggregation occurs?

*YOUR ANSWERS:*

a) 

b) 

c) 


---

## Exercise 2.4: Understanding Spark's Execution Plan

Spark has a query optimizer that creates an execution plan.

**Use `.explain(True)` to see:**
- Parsed logical plan (what you wrote)
- Analyzed logical plan (with schema info)
- Optimized logical plan (Spark's improvements)
- Physical plan (actual execution steps)

In [22]:
# Show the execution plan
print("Spark Execution Plan:")
print("=" * 80)
max_temps.explain(True)

Spark Execution Plan:
== Parsed Logical Plan ==
'Aggregate ['city], ['city, 'max('temperature) AS max_temperature#44]
+- Project [_c0#17 AS city#27, _c1#18 AS date#28, _c2#19 AS temperature#29]
   +- Relation [_c0#17,_c1#18,_c2#19] csv

== Analyzed Logical Plan ==
city: string, max_temperature: int
Aggregate [city#27], [city#27, max(temperature#29) AS max_temperature#44]
+- Project [_c0#17 AS city#27, _c1#18 AS date#28, _c2#19 AS temperature#29]
   +- Relation [_c0#17,_c1#18,_c2#19] csv

== Optimized Logical Plan ==
Aggregate [city#27], [city#27, max(temperature#29) AS max_temperature#44]
+- Project [_c0#17 AS city#27, _c2#19 AS temperature#29]
   +- Relation [_c0#17,_c1#18,_c2#19] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[city#27], functions=[max(temperature#29)], output=[city#27, max_temperature#44])
   +- Exchange hashpartitioning(city#27, 200), ENSURE_REQUIREMENTS, [plan_id=124]
      +- HashAggregate(keys=[city#27], functions=[partial_max(

---

### Question 8

**Look at the physical plan output above and answer:**

a) How many stages does Spark create?

b) Can you identify where the shuffle (Exchange) happens?

c) Why does Spark need a shuffle for this operation?

*Hint: Look for keywords like "Exchange", "HashAggregate", "Scan" in the physical plan.*

*YOUR ANSWERS:*

a) 

b) 

c) 


---

## Exercise 2.5: Caching for Performance

This is where you'll see Spark's advantage over Hadoop!

**Experiment:**
1. Cache the DataFrame in memory
2. Run the aggregation twice
3. Compare execution times

**The difference:** 
- Hadoop writes to disk after every operation
- Spark keeps data in RAM between operations

In [23]:
import time

# Cache the DataFrame
df_cached = df.cache()

# First execution - will load data into memory
print("First execution (loading into cache)...")
start_time = time.time()

max_temps_1 = df_cached.groupBy("city").agg(
    F.max("temperature").alias("max_temperature")
)
max_temps_1.show()  # Action triggers execution

first_duration = time.time() - start_time
print(f"\n⏱ First execution took: {first_duration:.3f} seconds")

First execution (loading into cache)...
+-------------+---------------+
|         city|max_temperature|
+-------------+---------------+
|  Los Angeles|             75|
|San Francisco|             62|
|     New York|             35|
+-------------+---------------+


⏱ First execution took: 0.155 seconds


In [24]:
# Second execution - should be faster using cached data
print("Second execution (using cached data)...")
start_time = time.time()

max_temps_2 = df_cached.groupBy("city").agg(
    F.max("temperature").alias("max_temperature")
)
max_temps_2.show()  # Action triggers execution

second_duration = time.time() - start_time
print(f"\n⏱ Second execution took: {second_duration:.3f} seconds")

# Calculate speedup
if second_duration > 0:
    speedup = first_duration / second_duration
    print(f"\nSpeedup: {speedup:.1f}x faster!")
else:
    print("\n⚡ Second execution was essentially instant!")

Second execution (using cached data)...
+-------------+---------------+
|         city|max_temperature|
+-------------+---------------+
|  Los Angeles|             75|
|San Francisco|             62|
|     New York|             35|
+-------------+---------------+


⏱ Second execution took: 0.085 seconds

Speedup: 1.8x faster!


---

### Question 9

**What happens the second time you run the aggregation on cached data? Why is it faster?**

*YOUR ANSWER:*



---

## Exercise 2.6: Challenge - Multi-Metric Analysis (Optional)

Extend your analysis to calculate multiple metrics at once!

**Calculate for each city:**
- Maximum temperature
- Minimum temperature
- Average temperature
- Count of days recorded

**Hint:** You can pass multiple aggregation functions to `.agg()`

In [25]:
# TODO: Calculate multiple metrics per city
# Use agg() with multiple functions like:
#   F.max("temperature").alias("max_temp"),
#   F.min("temperature").alias("min_temp"),
#   F.avg("temperature").alias("avg_temp"),
#   F.count("temperature").alias("num_days")

multi_metrics = df.groupBy("city").agg(
    # Add your aggregation functions here
)

print("Multi-metric analysis results:")
# multi_metrics.show()

AssertionError: exprs should not be empty

---

## Exercise 2.7: Explore Spark UI (Optional - 5 minutes)

While your SparkSession is running, open http://localhost:4040 in your browser.

1. Click on the **Jobs** tab - how many jobs did your notebook create?

2. Click on a job, then click on a **Stage** - can you see:
   - Number of tasks?
   - Input/output sizes?
   - Time spent?

3. Click on the **SQL** tab - find your groupBy query and look at the visual DAG.
   - Can you identify the Exchange (shuffle) in the diagram?

This is how you debug and optimize Spark jobs in production!

---

## Cleanup

Stop Spark and clean up files.

In [26]:
# Stop Spark
spark.stop()
print("✓ Spark session stopped")

✓ Spark session stopped


In [ ]:
# Optional: Clean up HDFS and local files
# Uncomment the lines below to run cleanup

# !hdfs dfs -rm -r /user/$(whoami)/week2/

# import os
# for f in ['sample.txt', 'downloaded_sample.txt', 'weather_data.txt']:
#     if os.path.exists(f):
#         os.remove(f)
#         print(f"Removed {f}")

---

# Part 3: Reflection Questions

Answer these deeper questions to solidify your understanding of distributed systems concepts.

---

### Question 10: HDFS Replication

- Why does HDFS default to 3x replication in production instead of 2x or 4x?
- In our lab, we used replication=1 for single-machine setup. Why is this acceptable for learning but not for production?
- What are the trade-offs between different replication levels?
- How does replication relate to fault tolerance?

*YOUR ANSWER:*



---

### Question 11: Data Locality

- How does HDFS enable data locality for MapReduce?
- Why is this critical for performance at petabyte scale?
- What would happen if computation always ran on different machines than data?

*YOUR ANSWER:*



---

### Question 12: MapReduce vs Spark

- When would you choose Hadoop MapReduce over Spark?
- When would you choose Spark over Hadoop?
- What's the fundamental architectural difference?

*YOUR ANSWER:*



---

### Question 13: Shuffle Phase

- Why is shuffle the most expensive phase in MapReduce?
- How does Spark optimize shuffle compared to Hadoop?
- What causes a shuffle operation in Spark?

*YOUR ANSWER:*



---

# Summary

**What you've learned:**

1. ✅ How to use HDFS CLI commands to interact with distributed storage
2. ✅ How blocks and replication work in HDFS
3. ✅ How to initialize Spark and read data from HDFS
4. ✅ How to perform group-by aggregations (MapReduce-style)
5. ✅ How to analyze Spark's execution plans
6. ✅ The performance impact of caching (Spark's advantage over Hadoop)

**Key Takeaways:**
- HDFS splits files into 128MB blocks, replicated across nodes for fault tolerance
- Data locality means moving compute to data, not data to compute
- Spark's lazy evaluation delays execution until an action is called
- Caching keeps data in RAM between operations (10-100x faster than disk)
- GroupBy operations trigger shuffles (expensive network operations)

---

## Submission

**Export this notebook to PDF:**
- File → Save and Export Notebook As... → PDF
- Or: `jupyter nbconvert --to pdf CS570_Week2_Lab.ipynb`

**Submit ONE PDF** containing all your code outputs and answers to Questions 1-13.

**Due:** Next class session

---

**Great work! 🎉**